In [ ]:
from modeling_distillemb import BertModel, BertForSequenceClassification, BertForEmbeddingLM
from distill_emb import DistillEmbSmall, DistillEmb
from config import DistillModelConfig, DistillEmbConfig
import torch
from transformers import AutoTokenizer, RwkvConfig, RwkvModel, AutoModel
from tokenizer import CharTokenizer
from knn_classifier import KNNTextClassifier
from data_loader import load_sentiment, load_ner_dataset, load_pos_dataset
from data_loader import load_news_dataset
import pandas as pd
from retrieval import build_json_pairs, top1_accuracy
import os
from transformers import GPT2LMHeadModel

In [ ]:
num_input_chars=12

In [ ]:
tokenizer = CharTokenizer.from_pretrained(pretrained_directory="logs/distil-emb-base")
distill_config = DistillEmbConfig.from_pretrained(pretrained_model_name_or_path="logs/distil-emb-base")
distill_model = DistillEmb.from_pretrained(pretrained_model_name_or_path="logs/distil-emb-base")

In [ ]:
distill_config

In [ ]:
# distill_config.distill_dropout = 0.25
config = DistillModelConfig(
    vocab_size=30522,
    hidden_size=512,
    num_hidden_layers=1,
    num_attention_heads=8,
    intermediate_size=3072,
    max_position_embeddings=1024,
    type_vocab_size=2,
    pad_token_id=0,
    position_embedding_type="absolute",
    use_cache=True,
    classifier_dropout=None,
    hidden_dropout_prob=0.1,
    embedding_type="distill",  # 'distilemb', 'fasttext'
    encoder_type='lstm', #'lstm'
    num_input_chars=num_input_chars,  # number of characters in each token
    char_vocab_size=tokenizer.char_vocab_size,
    distill_config=distill_config,
    distill_pretrained_model_name="logs/distil-emb-base",
    is_decoder=False
)


In [ ]:
path = "downstream-data/sentiment.parquet"
df = pd.read_parquet(path)
if 'sent' in path:
    # remove 0th index
    df = df[df['text'] != 'tweet'].reset_index(drop=True)

In [ ]:
df

In [ ]:
len(df['lang'].unique())

In [ ]:
lang_counts = df.groupby('split')['lang'].nunique()
for split, count in lang_counts.items():
    print(f"{split.capitalize()} split has {count} languages.")

In [ ]:
label2id = {label: idx for idx, label in enumerate(sorted(df['label'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}

df['label'] = df['label'].map(label2id).astype(int)

config.label2id = label2id
config.id2label = id2label

print(f"Converted labels to integers: {label2id}")

In [ ]:
num_labels = len(df['label'].unique())
config.num_labels = num_labels
model = BertForSequenceClassification(config)

In [ ]:
# # input ids with (B, S, N)
# char_input = torch.randint(0, config.num_input_chars, (1, 10, config.num_input_chars))
# # input ids with (B, S, N)
# print("char_input shape:", char_input.shape)
# inputs = {
#     "input_ids": char_input,
#     "attention_mask":torch.tensor([[1] * char_input.size(1)]),  # attention mask for each token
#     "token_type_ids": torch.tensor([[0] * char_input.size(1)]),  # token type ids for each token
# }
# outputs = model(**inputs)

In [ ]:
# tokenizer("Hello world!")

In [ ]:
# outputs[0].shape

In [ ]:
# model.save_pretrained("distil-emb-seqcls-lstm")

In [ ]:
# model = BertForSequenceClassification.from_pretrained("distil-emb-seqcls-lstm")

In [ ]:
labels = [x.item() for x in df['label'].unique()]
print(labels)
text_col = 'text'

In [ ]:
from datasets import Dataset, DatasetDict
df['text'] = df[text_col]
# Assuming df is your dataframe
# Split the data based on the 'split' column
train_df = df[df['split'] == 'train'][['text', 'label']]
test_df = df[df['split'] == 'test'][['text', 'label']]

# Create HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
train_dataset

In [ ]:
from typing import Dict, Any

def preprocess_function(examples: Dict[str, Any]):
    batch = tokenizer(
        examples["text"],
        padding=False,
        max_length=256,
        return_attention_mask=False,
    )

    batch["labels"] = examples["label"]
    return batch



tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

In [ ]:
len(train_dataset[0]['text'].split())

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
class CustomDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch = self.tokenizer.pad(
            features,
            padding="longest",
            max_length=256,
            return_tensors="pt",
            return_attention_mask=True,
            padding_side="right"
        )
        return batch

data_collator = CustomDataCollator(tokenizer)

In [ ]:
##### from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted', labels=labels)
    f1_macro = f1_score(labels, predictions, average='macro', labels=labels)
    f1_micro = f1_score(labels, predictions, average='micro', labels=labels)
    return {"accuracy": acc, "f1_weighted": f1, "f1_macro": f1_macro, "f1_micro": f1_micro}


import os
dataloader_num_workers=os.cpu_count() - 1
batch_size = 32

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=15,
    weight_decay=0.01,
    report_to=[],
    eval_strategy="epoch",  
    save_total_limit=1,
    save_only_model=True,
    logging_strategy="steps",
    logging_steps=10,
    label_smoothing_factor=0.1,
    max_grad_norm=5.0,
    warmup_ratio=0.0,
    lr_scheduler_type="cosine",
    dataloader_num_workers=16,        # Number of CPU workers for data loading
    dataloader_pin_memory=True,      # Faster GPU transfer
    gradient_accumulation_steps=4
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Evaluate the model after training
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

In [ ]:
trainer.evaluate()

In [ ]:
# model = BertForSequenceClassification.from_pretrained("distil-emb-seqcls-lstm").cuda()
model = trainer.model
model.eval()
# Ensure 'language' column exists in df
test_df = df[df['split'] == 'test'][['text', 'label', 'lang']]
languages = test_df['lang'].unique()
per_language_f1 = {}

batch_size = 16

for lang in languages:
    lang_df = test_df[test_df['lang'] == lang]
    texts = lang_df['text'].tolist()
    labels = lang_df['label'].values
    preds = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]
        tokenized = tokenizer(
            batch_texts,
            padding='longest',
            truncation=True,
            max_length=256,
            return_tensors="pt",
            return_attention_mask=True,
            padding_side="right"
        )
        with torch.no_grad():
            inputs = {k: v.cuda() for k, v in tokenized.items()}
            outputs = model(**inputs)
            batch_preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            preds.extend(batch_preds)
    f1 = f1_score(labels, preds, average='weighted', labels=labels)
    per_language_f1[lang] = f1

# Print per-language F1
for lang, f1 in per_language_f1.items():
    print(f"Language: {lang}, F1: {f1:.4f}")

# Average F1
average_f1 = sum(per_language_f1.values()) / len(per_language_f1)
print(f"Average F1 across languages: {average_f1:.4f}")

In [ ]:
model.save_pretrained("distil-emb-news-lstm-best-256")